# 01 — Data Ingestion

## Purpose

Build a reproducible corpus of World Bank Project Appraisal Documents (PADs)
for innovation analysis.

This notebook:

1. Queries the World Bank Documents & Reports API.
2. Filters PADs by period, region, and sector.
3. Creates a reproducible prototype sample.
4. Downloads machine-readable project text.
5. Saves metadata manifests for downstream processing.

In [ ]:
# --------------------------------------------------
# IMPORTS AND PATHS
# --------------------------------------------------

import json
import time
from pathlib import Path

import pandas as pd
import requests

ROOT          = Path("..").resolve()
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
META_DIR      = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"
PAD_DIR       = RAW_DIR / "pad"
ICR_DIR       = RAW_DIR / "icr"
OUTPUT_DIR    = ROOT / "outputs"

for folder in [DATA_DIR, RAW_DIR, META_DIR, PROCESSED_DIR,
               PAD_DIR, ICR_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BASE_URL       = "https://search.worldbank.org/api/v3/wds"
PAD_TYPE       = "Project Appraisal Document"
ICR_TYPE       = "Implementation Completion and Results Report"
PROTOTYPE_SIZE = 60
RANDOM_SEED    = 42

print("Project root:", ROOT)

In [ ]:
# --------------------------------------------------
# API HELPERS
# --------------------------------------------------

def query_wb_api(params, timeout=60, verbose=False):
    """Query the World Bank Documents and Reports API."""
    query_params = params.copy()
    query_params.setdefault("format", "json")
    response = requests.get(BASE_URL, params=query_params, timeout=timeout)
    if verbose:
        print("Status:", response.status_code)
        print("URL   :", response.url)
    response.raise_for_status()
    return response.json()


def extract_document_records(api_response):
    """Extract document records from a WB API response (keys starting with D)."""
    documents = api_response.get("documents", {})
    return [rec for key, rec in documents.items() if key.startswith("D")]


def fetch_all_documents(base_params, page_size=100):
    """Paginate through all matching WB document records."""
    all_records = []
    offset = 0
    while True:
        params = {**base_params, "rows": page_size, "os": offset}
        data = query_wb_api(params)
        records = extract_document_records(data)
        all_records.extend(records)
        total = data["total"]
        print(f"Fetched {len(all_records):,} of {total:,} records")
        if len(all_records) >= total or not records:
            break
        offset += page_size
    return all_records

In [ ]:
# --------------------------------------------------
# FETCH FULL PAD CORPUS METADATA
# --------------------------------------------------

all_pad_records = fetch_all_documents(
    {
        "docty_exact": PAD_TYPE,
        "strdate": "2015-01-01",
        "enddate": "2025-12-31",
    },
    page_size=100,
)

all_pad_df = pd.DataFrame(all_pad_records)

print("PAD metadata shape:", all_pad_df.shape)
print("Unique projects   :", all_pad_df["projectid"].nunique())

In [ ]:
# --------------------------------------------------
# SSA FILTER
# --------------------------------------------------
#Define and apply the Sub-Saharan Africa filter



SSA_REGION_TERMS = ["Eastern and Southern Africa", "Western and Central Africa"]


def is_ssa_region(region):
    if pd.isna(region):
        return False
    if region == "Africa":
        return True
    return any(term in region for term in SSA_REGION_TERMS)


ssa_pad_df = all_pad_df[all_pad_df["admreg"].apply(is_ssa_region)].copy()

print(f"SSA PAD records: {len(ssa_pad_df):,} of {len(all_pad_df):,} total")
print()
print(ssa_pad_df["admreg"].value_counts(dropna=False))

In [ ]:
# --------------------------------------------------
# SECTOR NORMALIZATION AND EXPLORATORY KEYWORD SIGNALS
# --------------------------------------------------

def extract_sector_names(value):
    if not isinstance(value, dict):
        return []
    return [item["sector"] for item in value.values()
            if isinstance(item, dict) and "sector" in item]


def normalize_sector_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    if name.startswith("FY17 - "):
        name = name.replace("FY17 - ", "", 1)
    return name


ssa_pad_df["sector_list"] = ssa_pad_df["sectr"].apply(extract_sector_names)
ssa_pad_df["sector_list_normalized"] = ssa_pad_df["sector_list"].apply(
    lambda sectors: [normalize_sector_name(s) for s in sectors]
)

DOMAIN_SECTORS = {
    "Agriculture, Fishing and Forestry",
    "Water, Sanitation and Waste Management",
    "Information and Communications Technologies",
    "Digital Development",
}

DOMAIN_KEYWORDS = {
    "irrigation":               ["irrigation", "drainage", "agricultural water"],
    "water_resources":          ["water resources", "water security", "watershed",
                                 "groundwater", "river basin", "hydrology", "water management"],
    "digital_data":             ["digital", "data", "information system",
                                 "information systems", "ict", "platform", "digital development"],
    "monitoring_remote_sensing":["remote sensing", "satellite", "earth observation",
                                 "monitoring", "geospatial", "gis"],
    "climate_resilience":       ["climate", "resilience", "drought", "flood", "climate-smart"],
}


def has_target_sector(sectors):
    return any(s in DOMAIN_SECTORS for s in sectors)


def combine_metadata_text(row):
    fields = [row.get("display_title", ""), row.get("subsc", ""), row.get("theme", "")]
    return " ".join(str(v) for v in fields if pd.notna(v)).lower()


def match_keyword_family(text, keywords):
    return any(kw.lower() in text for kw in keywords)


domain_candidate_df = ssa_pad_df[
    ssa_pad_df["sector_list_normalized"].apply(has_target_sector)
].copy()

domain_candidate_df["metadata_text"] = domain_candidate_df.apply(
    combine_metadata_text, axis=1
)

for family, keywords in DOMAIN_KEYWORDS.items():
    domain_candidate_df[f"match_{family}"] = (
        domain_candidate_df["metadata_text"]
        .apply(lambda text: match_keyword_family(text, keywords))
    )

match_columns = [f"match_{f}" for f in DOMAIN_KEYWORDS]
domain_candidate_df["keyword_match_count"] = (
    domain_candidate_df[match_columns].sum(axis=1)
)

print(f"Broad domain candidates: {len(domain_candidate_df):,} of {len(ssa_pad_df):,} SSA PADs")
print()
print(domain_candidate_df["keyword_match_count"].value_counts().sort_index())

In [ ]:
# --------------------------------------------------
# WORKING CORPUS AND MANIFEST
# --------------------------------------------------

working_pad_df = (
    domain_candidate_df
    .sort_values(
        ["projectid", "docdt", "last_modified_date"],
        ascending=[True, False, False]
    )
    .drop_duplicates(subset=["projectid", "guid"], keep="first")
    .reset_index(drop=True)
)

working_pad_df["has_text"]     = working_pad_df["txturl"].notna()
working_pad_df["has_pdf"]      = working_pad_df["pdfurl"].notna()
working_pad_df["document_year"] = pd.to_datetime(
    working_pad_df["docdt"], errors="coerce"
).dt.year

print("Working PAD records:", len(working_pad_df))
print("Unique projects    :", working_pad_df["projectid"].nunique())
print("Text available     :", working_pad_df["has_text"].sum())

working_pad_df.to_csv(META_DIR / "pad_corpus_manifest.csv", index=False)
print("Saved: pad_corpus_manifest.csv")

In [ ]:
# --------------------------------------------------
# DOWNLOAD PROTOTYPE SAMPLE
# --------------------------------------------------

def download_text_document(row, output_dir):
    url = row["txturl"]
    if pd.isna(url):
        return "missing_url"
    filename    = f"{row['projectid']}_{row['guid']}.txt"
    output_path = output_dir / filename
    if output_path.exists():
        return "exists"
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        output_path.write_text(response.text, encoding="utf-8")
        return "downloaded"
    except requests.RequestException:
        return "failed"


download_sample_df = (
    working_pad_df[working_pad_df["has_text"]]
    .sample(n=min(PROTOTYPE_SIZE, working_pad_df["has_text"].sum()),
            random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print("Documents selected:", len(download_sample_df))

download_results = [
    download_text_document(row, PAD_DIR)
    for _, row in download_sample_df.iterrows()
]

download_sample_df["download_status"] = download_results

print(pd.Series(download_results).value_counts().to_string())

download_sample_df.to_csv(META_DIR / "prototype_pad_sample.csv", index=False)
print("Saved: prototype_pad_sample.csv")